# RCEA Score Sensitivity Analysis

This notebook examines how RCEA subscores change as the finding's uncertainty level
and evidence level vary, using the privacy/recruitment example.


In [ ]:
import sys
sys.path.insert(0, '..')

import json
from pathlib import Path
from rcea.models import EvidenceFinding, ContextPack, RoleProfile, RulePack, Uncertainty, EvidenceLevel
from rcea.passport import generate_passport

BASE = Path('../examples/privacy_recruitment')
finding_base = EvidenceFinding.model_validate_json((BASE / 'finding.json').read_text())
context      = ContextPack.model_validate_json((BASE / 'context_pack.json').read_text())
roles_data   = json.loads((BASE / 'role_profiles.json').read_text())
roles        = {r['role_id']: RoleProfile.model_validate(r) for r in roles_data}
rule_pack    = RulePack.model_validate_json((BASE / 'rule_pack.json').read_text())

In [ ]:
# Sweep uncertainty levels for DPO
role = roles['dpo']
uncertainties = ['low', 'medium', 'high', 'unknown']

print(f"Sensitivity to uncertainty (role=dpo)")
print(f"{'Uncertainty':<12} {'EW':>8} {'Overall':>10}")
print('-' * 32)
for unc in uncertainties:
    variant = finding_base.model_copy(update={'uncertainty': Uncertainty(unc)})
    passport = generate_passport(variant, role, context, rule_pack)
    ew = passport.rcea_scores.get('epistemic_warrant', float('nan'))
    print(f"{unc:<12} {ew:>8.4f} {passport.overall_rcea:>10.4f}")

In [ ]:
# Sweep evidence levels for DPO
evidence_levels = ['assertion', 'documented', 'tested', 'independently_verified']

print(f"Sensitivity to evidence_level (role=dpo)")
print(f"{'Evidence Level':<28} {'EW':>8} {'Overall':>10}")
print('-' * 48)
for level in evidence_levels:
    variant = finding_base.model_copy(update={'evidence_level': EvidenceLevel(level)})
    passport = generate_passport(variant, role, context, rule_pack)
    ew = passport.rcea_scores.get('epistemic_warrant', float('nan'))
    print(f"{level:<28} {ew:>8.4f} {passport.overall_rcea:>10.4f}")

In [ ]:
# Cross-role sensitivity: same finding, all roles
print(f"Cross-role RCEA scores (same finding)")
print(f"{'Role':<14} {'MR':>7} {'EW':>7} {'NA':>7} {'IF':>7} {'DA':>7} {'LP':>7} {'AT':>7} {'Overall':>9}")
print('-' * 75)
keys = ['material_relevance','epistemic_warrant','normative_alignment',
        'interpretive_fit','decision_actionability','limitation_propagation','audit_traceability']
for role_id, role in roles.items():
    passport = generate_passport(finding_base, role, context, rule_pack)
    s = passport.rcea_scores
    vals = ''.join(f"{s.get(k, float('nan')):>7.3f}" for k in keys)
    print(f"{role_id:<14}{vals}{passport.overall_rcea:>9.4f}")